# Setup

In [1]:
! python -m pip install --no-index --find-links=../input/packges-offline -r ../input/packges-offline/requirements.txt

Looking in links: ../input/packges-offline
Processing /kaggle/input/packges-offline/fedot-0.7.5-py3-none-any.whl (from -r ../input/packges-offline/requirements.txt (line 1))
Processing /kaggle/input/packges-offline/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl (from -r ../input/packges-offline/requirements.txt (line 2))
Processing /kaggle/input/packges-offline/scipy-1.12.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from fedot->-r ../input/packges-offline/requirements.txt (line 1))
Processing /kaggle/input/packges-offline/thegolem-0.4.1-py3-none-any.whl (from fedot->-r ../input/packges-offline/requirements.txt (line 1))
Processing /kaggle/input/packges-offline/anytree-2.13.0-py3-none-any.whl (from fedot->-r ../input/packges-offline/requirements.txt (line 1))
Processing /kaggle/input/packges-offline/ete3-3.1.3.tar.gz (from fedot->-r ../input/packges-offline/requirements.txt (line 1))
  Preparing metadata (setup.py) ... done
Processing /kaggle/input/packges-offline

# Splitting data

In [2]:
import os
import pandas as pd
import shutil

# Define paths
source_dir = "/kaggle/input/neurips-open-polymer-prediction-2025"
base_dir = "/kaggle/working"
targets = ["Tg", "FFV", "Tc", "Density", "Rg"]

# Load files once
train_df = pd.read_csv(os.path.join(source_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(source_dir, "test.csv"))
sample_df = pd.read_csv(os.path.join(source_dir, "sample_submission.csv"))

# Loop through each target
for target in targets:
    # Create output directory
    target_dir = os.path.join(base_dir, f"polymer_{target}", "competition",)
    os.makedirs(target_dir, exist_ok=True)

    # Copy test file as-is
    test_df.to_csv(os.path.join(target_dir, "test.csv"), index=False)

    # Prepare sample_submission with only current target
    sample_target_df = sample_df[["id", target]]
    sample_target_df.to_csv(os.path.join(target_dir, "sample_submission.csv"), index=False)

    # Prepare train file: only rows where target is not null, and only Id + target columns
    train_target_df = train_df[["id","SMILES", target]].dropna(subset=[target])
    train_target_df.to_csv(os.path.join(target_dir, "train.csv"), index=False)

# Evaluating in 5 separate runs

In [3]:
### UNMODIFIABLE IMPORT BEGIN ###
import random
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Tuple
from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import (
    Task,
    TaskTypesEnum,
)  # classification, regression, ts_forecasting.
def train_model(train_features: np.ndarray | pd.DataFrame, train_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(train_features, pd.DataFrame) and isinstance(train_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(train_features, train_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(train_features, np.ndarray) and isinstance(train_target, np.ndarray):
        input_data = InputData.from_numpy(train_features, train_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for train_features and train_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(train_features)} and {type(train_target)}")
        
    model = Fedot(problem=TaskTypesEnum.regression.value,
            timeout=1,
            seed=42,
            cv_folds=5,
            preset='auto',
            metric='mae',
            n_jobs=1,
            with_tuning=True,
            show_progress=True)

    try:
        model.fit(features=input_data) # this is the training step, after this step variable 'model' will be a trained model
    except Exception as e:
        raise RuntimeError(
            f"Model training failed. Please check your data preprocessing carefully. "
            f"Common issues include: missing values, incorrect data types, feature scaling problems, "
            f"or incompatible target variable format. Original error: {str(e)}"
        ) from e

    # Save the pipeline
    pipeline = model.current_pipeline
    pipeline.save(path=PIPELINE_PATH, create_subdir=False, is_datetime_in_path=False)

    return model
def evaluate_model(model: Fedot, test_features: np.ndarray | pd.DataFrame | pd.Series, test_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(test_features, pd.DataFrame) and isinstance(test_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(test_features, np.ndarray) and isinstance(test_target, np.ndarray):
        input_data = InputData.from_numpy(test_features, test_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for test_features and test_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(test_features)} and {type(test_target)}")
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()
def automl_predict(model: Fedot, features: np.ndarray | pd.DataFrame | pd.Series) -> np.ndarray:
    if isinstance(features, (pd.DataFrame, pd.Series)):
        features = features.to_numpy()
    input_data = InputData.from_numpy(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")
    return predictions
def smiles_to_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            "MolWt": Descriptors.MolWt(mol)
            ,"HeavyAtomMolWt": Descriptors.HeavyAtomMolWt(mol)
            ,"NumAtoms": mol.GetNumAtoms()
            ,"MolLogP": Descriptors.MolLogP(mol)
            ,"NumHDonors": Descriptors.NumHDonors(mol)
            ,"NumHAcceptors": Descriptors.NumHAcceptors(mol)
            ,"TPSA": Descriptors.TPSA(mol)
            ,"NumRotatableBonds": Descriptors.NumRotatableBonds(mol)
            ,"RingCount": Descriptors.RingCount(mol)
            ,"NumAromaticRings": Descriptors.NumAromaticRings(mol)
            ,"NumHeteroatoms": Descriptors.NumHeteroatoms(mol)
        }
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fp = mfpgen.GetFingerprint(mol)
        features.update({f'FP_{i}': int(b) for i, b in enumerate(fp)})
        
        return features
    except:
        return None

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

### UNMODIFIABLE IMPORT END ###
# USER CODE BEGIN IMPORTS #
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
# USER CODE END IMPORTS #

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### UNMODIFIABLE CODE BEGIN ###
DATASET_PATH = Path("/kaggle/working/polymer_Density/competition")  # path for saving and loading dataset(s)
WORKSPACE_PATH = Path("/kaggle/working/polymer_Density/output")
PIPELINE_PATH = WORKSPACE_PATH / "pipeline"  # path for saving and loading pipelines
SUBMISSION_PATH = WORKSPACE_PATH / "submission.csv"  # path for saving submission file
EVAL_SET_SIZE = 0.2  # 20% of the data for evaluation
### UNMODIFIABLE CODE END ###

# --- TODO: Update these paths for your specific competition ---
TRAIN_FILE = DATASET_PATH / "train.csv"  # TODO: Replace with your actual filename
TEST_FILE = DATASET_PATH / "test.csv"  # TODO: Replace with your actual filename
SAMPLE_SUBMISSION_FILE = DATASET_PATH / "sample_submission.csv"  # TODO: Replace with your actual filename or None


# USER CODE BEGIN LOAD_DATA #
def load_data():
    train = pd.read_csv(TRAIN_FILE)
    X_test = pd.read_csv(TEST_FILE)
    return train, X_test
# USER CODE END LOAD_DATA #

def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Function to transform data into a format that can be used for training the model.
    Used on both Train and Test data. Test data may initially not contain target columns.
    """

    target_columns = ['Density']  # TODO: Replace with ACTUAL target columns

    # Separating features and target if present
    data = dataset.copy(deep=True)

    features = data['SMILES'].apply(smiles_to_features)
    valid_smiles = features.notna()
    if not valid_smiles.any():
        raise ValueError("No valid SMILES strings found")
    data = data[valid_smiles].drop(columns=['SMILES'])
    features = features[valid_smiles].apply(pd.Series)
    data = pd.concat([data, features], axis=1)

    has_target = any(col in data.columns for col in target_columns)
    if has_target:
        features = data.drop(columns=target_columns)
        target = data[target_columns].values.flatten()  # Flatten target to 1-D array for regression
    else:
        features = data
        target = None

    return features.values, target


# The main function to orchestrate the data loading, feature engineering, model training and model evaluation
def create_model():
    """
    Function to execute the ML pipeline.
    """
    # USER CODE BEGIN CREATE MODEL #
    # Step 1. Retrieve or load a dataset from hub (if available) or user’s local storage, start path from the DATASET_PATH
    train, X_test = load_data()

    # Step 2. Create a train-test split of the data by splitting the ‘dataset‘ into train_data and test_data.
    # Create a train-validation split
    # Note: EVAL_SET_SIZE is a constant defined above, corresponding to 20% of the data for evaluation
    # Note: You may need to use stratified sampling if the target is categorical
    train_data, eval_test_data = train_test_split(
        train, test_size=EVAL_SET_SIZE, random_state=SEED
    )  # corresponding to 80%, 20% of ‘dataset‘

    train_features, train_target = transform_data(train_data)
    eval_test_features, eval_test_target = transform_data(eval_test_data)
    test_features, _ = transform_data(X_test)

    # Step 3. Train AutoML model. AutoML performs feature engineering and model training.
    model = train_model(train_features, train_target)

    # Step 4. evaluate the trained model using the defined "evaluate_model" function
    model_performance = evaluate_model(model, eval_test_features, eval_test_target)

    # Step 5.  Evaluate predictions for the test dataset
    # **YOU MUST USE automl_predict()**
    predictions: np.ndarray = automl_predict(model, test_features)  # returns 2D array
    output = pd.DataFrame(predictions, columns=['Density'])  # Adding column name for predictions

    # Adding IDs to output
    output.insert(0, 'id', X_test['id'].values)  # Insert ID column
    output['id'] = output['id'].astype(int)  # Ensure ID column is of integer type

    # USER CODE END CREATE MODEL #
    # If target submission format is not numeric, convert predictions to expected format
    output.to_csv(SUBMISSION_PATH, index=False)
    return model_performance


### UNMODIFIABLE CODE BEGIN ###
def main():
    """
    Main function to execute the ML pipeline.
    """
    print("Files and directories:")
    paths = {
        "Dataset Path": DATASET_PATH,
        "Workspace Path": WORKSPACE_PATH,
        "Pipeline Path": PIPELINE_PATH,
        "Submission Path": SUBMISSION_PATH,
        "Train File": TRAIN_FILE,
        "Test File": TEST_FILE,
        "Sample Submission File": SAMPLE_SUBMISSION_FILE,
    }
    for name, path in paths.items():
        print(f"{name}: {path}")

    model_performance = create_model()
    print("Model Performance on Test Set:", model_performance)


main()

2025-07-24 00:28:04,115 - Enabling RDKit 2025.03.3 jupyter extensions
Files and directories:
Dataset Path: /kaggle/working/polymer_Density/competition
Workspace Path: /kaggle/working/polymer_Density/output
Pipeline Path: /kaggle/working/polymer_Density/output/pipeline
Submission Path: /kaggle/working/polymer_Density/output/submission.csv
Train File: /kaggle/working/polymer_Density/competition/train.csv
Test File: /kaggle/working/polymer_Density/competition/test.csv
Sample Submission File: /kaggle/working/polymer_Density/competition/sample_submission.csv


2025-07-24 00:28:33.718035: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753316913.939269      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753316914.003944      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2025-07-24 00:29:29,953 - Topological features operation requires extra dependencies for time series forecasting, which are not installed. It can infuence the performance. Please install it by 'pip install fedot[extra]'
2025-07-24 00:29:42,510 - Unknown integration target: 
2025-07-24 00:29:45,382 - Unknown integration target: 
2025-07-24 00:29:49,496 - Unknown integration target: 
2025-07-24 00:29:52,076 - Unknown integration target: 
2025-07-24 00:29:55,213 - Unknown integration target: 
2025-07-24 00:30:01,194 - Unknown integration target: 
2025-07-24 00:30:07,001 - Unknown integration target: 
2025-07-24 00:30:14,002 - Unknown integration target: 
2025-07-24 00:31:25,799 - Unknown integration target: 
2025-07-24 00:31:25,801 - Unknown integration target: 
2025-07-24 00:31:34,279 - ApiComposer - Initial pipeline was fitted in 7.3 sec.
2025-07-24 00:31:34,282 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 36.7 sec.
2025-07-24 00:31:34,284 

In [4]:
### UNMODIFIABLE IMPORT BEGIN ###
import random
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Tuple
from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import (
    Task,
    TaskTypesEnum,
)  # classification, regression, ts_forecasting.
def train_model(train_features: np.ndarray | pd.DataFrame, train_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(train_features, pd.DataFrame) and isinstance(train_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(train_features, train_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(train_features, np.ndarray) and isinstance(train_target, np.ndarray):
        input_data = InputData.from_numpy(train_features, train_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for train_features and train_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(train_features)} and {type(train_target)}")
        
    model = Fedot(problem=TaskTypesEnum.regression.value,
            timeout=1,
            seed=42,
            cv_folds=5,
            preset='auto',
            metric='mae',
            n_jobs=1,
            with_tuning=True,
            show_progress=True)

    try:
        model.fit(features=input_data) # this is the training step, after this step variable 'model' will be a trained model
    except Exception as e:
        raise RuntimeError(
            f"Model training failed. Please check your data preprocessing carefully. "
            f"Common issues include: missing values, incorrect data types, feature scaling problems, "
            f"or incompatible target variable format. Original error: {str(e)}"
        ) from e

    # Save the pipeline
    pipeline = model.current_pipeline
    pipeline.save(path=PIPELINE_PATH, create_subdir=False, is_datetime_in_path=False)

    return model
def evaluate_model(model: Fedot, test_features: np.ndarray | pd.DataFrame | pd.Series, test_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(test_features, pd.DataFrame) and isinstance(test_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(test_features, np.ndarray) and isinstance(test_target, np.ndarray):
        input_data = InputData.from_numpy(test_features, test_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for test_features and test_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(test_features)} and {type(test_target)}")
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()
def automl_predict(model: Fedot, features: np.ndarray | pd.DataFrame | pd.Series) -> np.ndarray:
    if isinstance(features, (pd.DataFrame, pd.Series)):
        features = features.to_numpy()
    input_data = InputData.from_numpy(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")
    return predictions
def smiles_to_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            "MolWt": Descriptors.MolWt(mol)
            ,"HeavyAtomMolWt": Descriptors.HeavyAtomMolWt(mol)
            ,"NumAtoms": mol.GetNumAtoms()
            ,"MolLogP": Descriptors.MolLogP(mol)
            ,"NumHDonors": Descriptors.NumHDonors(mol)
            ,"NumHAcceptors": Descriptors.NumHAcceptors(mol)
            ,"TPSA": Descriptors.TPSA(mol)
            ,"NumRotatableBonds": Descriptors.NumRotatableBonds(mol)
            ,"RingCount": Descriptors.RingCount(mol)
            ,"NumAromaticRings": Descriptors.NumAromaticRings(mol)
            ,"NumHeteroatoms": Descriptors.NumHeteroatoms(mol)
        }
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fp = mfpgen.GetFingerprint(mol)
        features.update({f'FP_{i}': int(b) for i, b in enumerate(fp)})
        
        return features
    except:
        return None

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

### UNMODIFIABLE IMPORT END ###
# USER CODE BEGIN IMPORTS #
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
# USER CODE END IMPORTS #

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### UNMODIFIABLE CODE BEGIN ###
DATASET_PATH = Path("/kaggle/working/polymer_FFV/competition")  # path for saving and loading dataset(s)
WORKSPACE_PATH = Path("/kaggle/working/polymer_FFV/output")
PIPELINE_PATH = WORKSPACE_PATH / "pipeline"  # path for saving and loading pipelines
SUBMISSION_PATH = WORKSPACE_PATH / "submission.csv"  # path for saving submission file
EVAL_SET_SIZE = 0.2  # 20% of the data for evaluation
### UNMODIFIABLE CODE END ###

# --- TODO: Update these paths for your specific competition ---
TRAIN_FILE = DATASET_PATH / "train.csv"  # TODO: Replace with your actual filename
TEST_FILE = DATASET_PATH / "test.csv"  # TODO: Replace with your actual filename
SAMPLE_SUBMISSION_FILE = DATASET_PATH / "sample_submission.csv"  # TODO: Replace with your actual filename or None


# USER CODE BEGIN LOAD_DATA #
def load_data():
    train = pd.read_csv(TRAIN_FILE)
    X_test = pd.read_csv(TEST_FILE)
    return train, X_test
# USER CODE END LOAD_DATA #

def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Function to transform data into a format that can be used for training the model.
    Used on both Train and Test data. Test data may initially not contain target columns.
    """

    target_columns = ['FFV']  # TODO: Replace with ACTUAL target columns

    # Separating features and target if present
    data = dataset.copy(deep=True)

    features = data['SMILES'].apply(smiles_to_features)
    valid_smiles = features.notna()
    if not valid_smiles.any():
        raise ValueError("No valid SMILES strings found")
    data = data[valid_smiles].drop(columns=['SMILES'])
    features = features[valid_smiles].apply(pd.Series)
    data = pd.concat([data, features], axis=1)

    has_target = any(col in data.columns for col in target_columns)
    if has_target:
        features = data.drop(columns=target_columns)
        target = data[target_columns].values.flatten()  # Flatten target to 1-D array for regression
    else:
        features = data
        target = None

    return features.values, target


# The main function to orchestrate the data loading, feature engineering, model training and model evaluation
def create_model():
    """
    Function to execute the ML pipeline.
    """
    # USER CODE BEGIN CREATE MODEL #
    # Step 1. Retrieve or load a dataset from hub (if available) or user’s local storage, start path from the DATASET_PATH
    train, X_test = load_data()

    # Step 2. Create a train-test split of the data by splitting the ‘dataset‘ into train_data and test_data.
    # Create a train-validation split
    # Note: EVAL_SET_SIZE is a constant defined above, corresponding to 20% of the data for evaluation
    # Note: You may need to use stratified sampling if the target is categorical
    train_data, eval_test_data = train_test_split(
        train, test_size=EVAL_SET_SIZE, random_state=SEED
    )  # corresponding to 80%, 20% of ‘dataset‘

    train_features, train_target = transform_data(train_data)
    eval_test_features, eval_test_target = transform_data(eval_test_data)
    test_features, _ = transform_data(X_test)

    # Step 3. Train AutoML model. AutoML performs feature engineering and model training.
    model = train_model(train_features, train_target)

    # Step 4. evaluate the trained model using the defined "evaluate_model" function
    model_performance = evaluate_model(model, eval_test_features, eval_test_target)

    # Step 5.  Evaluate predictions for the test dataset
    # **YOU MUST USE automl_predict()**
    predictions: np.ndarray = automl_predict(model, test_features)  # returns 2D array
    output = pd.DataFrame(predictions, columns=['FFV'])  # Adding column name for predictions

    # Adding IDs to output
    output.insert(0, 'id', X_test['id'].values)  # Insert ID column
    output['id'] = output['id'].astype(int)  # Ensure ID column is of integer type

    # USER CODE END CREATE MODEL #
    # If target submission format is not numeric, convert predictions to expected format
    output.to_csv(SUBMISSION_PATH, index=False)
    return model_performance


### UNMODIFIABLE CODE BEGIN ###
def main():
    """
    Main function to execute the ML pipeline.
    """
    print("Files and directories:")
    paths = {
        "Dataset Path": DATASET_PATH,
        "Workspace Path": WORKSPACE_PATH,
        "Pipeline Path": PIPELINE_PATH,
        "Submission Path": SUBMISSION_PATH,
        "Train File": TRAIN_FILE,
        "Test File": TEST_FILE,
        "Sample Submission File": SAMPLE_SUBMISSION_FILE,
    }
    for name, path in paths.items():
        print(f"{name}: {path}")

    model_performance = create_model()
    print("Model Performance on Test Set:", model_performance)


main()

Files and directories:
Dataset Path: /kaggle/working/polymer_FFV/competition
Workspace Path: /kaggle/working/polymer_FFV/output
Pipeline Path: /kaggle/working/polymer_FFV/output/pipeline
Submission Path: /kaggle/working/polymer_FFV/output/submission.csv
Train File: /kaggle/working/polymer_FFV/competition/train.csv
Test File: /kaggle/working/polymer_FFV/competition/test.csv
Sample Submission File: /kaggle/working/polymer_FFV/competition/sample_submission.csv
2025-07-24 00:35:15,139 - ApiComposer - Initial pipeline was fitted in 50.5 sec.
2025-07-24 00:35:15,142 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 252.7 sec.
2025-07-24 00:35:15,144 - AssumptionsHandler - Preset was changed to fast_train due to fit time estimation for initial model.
2025-07-24 00:35:15,155 - ApiComposer - AutoML configured. Parameters tuning: True. Time limit: 1 min. Set of candidate models: ['adareg', 'dtreg', 'knnreg', 'lasso', 'linear', 'normalization', 'pca', 'ra

In [5]:
### UNMODIFIABLE IMPORT BEGIN ###
import random
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Tuple
from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import (
    Task,
    TaskTypesEnum,
)  # classification, regression, ts_forecasting.
def train_model(train_features: np.ndarray | pd.DataFrame, train_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(train_features, pd.DataFrame) and isinstance(train_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(train_features, train_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(train_features, np.ndarray) and isinstance(train_target, np.ndarray):
        input_data = InputData.from_numpy(train_features, train_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for train_features and train_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(train_features)} and {type(train_target)}")
        
    model = Fedot(problem=TaskTypesEnum.regression.value,
            timeout=1,
            seed=42,
            cv_folds=5,
            preset='auto',
            metric='mae',
            n_jobs=1,
            with_tuning=True,
            show_progress=True)

    try:
        model.fit(features=input_data) # this is the training step, after this step variable 'model' will be a trained model
    except Exception as e:
        raise RuntimeError(
            f"Model training failed. Please check your data preprocessing carefully. "
            f"Common issues include: missing values, incorrect data types, feature scaling problems, "
            f"or incompatible target variable format. Original error: {str(e)}"
        ) from e

    # Save the pipeline
    pipeline = model.current_pipeline
    pipeline.save(path=PIPELINE_PATH, create_subdir=False, is_datetime_in_path=False)

    return model
def evaluate_model(model: Fedot, test_features: np.ndarray | pd.DataFrame | pd.Series, test_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(test_features, pd.DataFrame) and isinstance(test_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(test_features, np.ndarray) and isinstance(test_target, np.ndarray):
        input_data = InputData.from_numpy(test_features, test_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for test_features and test_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(test_features)} and {type(test_target)}")
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()
def automl_predict(model: Fedot, features: np.ndarray | pd.DataFrame | pd.Series) -> np.ndarray:
    if isinstance(features, (pd.DataFrame, pd.Series)):
        features = features.to_numpy()
    input_data = InputData.from_numpy(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")
    return predictions
def smiles_to_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            "MolWt": Descriptors.MolWt(mol)
            ,"HeavyAtomMolWt": Descriptors.HeavyAtomMolWt(mol)
            ,"NumAtoms": mol.GetNumAtoms()
            ,"MolLogP": Descriptors.MolLogP(mol)
            ,"NumHDonors": Descriptors.NumHDonors(mol)
            ,"NumHAcceptors": Descriptors.NumHAcceptors(mol)
            ,"TPSA": Descriptors.TPSA(mol)
            ,"NumRotatableBonds": Descriptors.NumRotatableBonds(mol)
            ,"RingCount": Descriptors.RingCount(mol)
            ,"NumAromaticRings": Descriptors.NumAromaticRings(mol)
            ,"NumHeteroatoms": Descriptors.NumHeteroatoms(mol)
        }
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fp = mfpgen.GetFingerprint(mol)
        features.update({f'FP_{i}': int(b) for i, b in enumerate(fp)})
        
        return features
    except:
        return None

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

### UNMODIFIABLE IMPORT END ###
# USER CODE BEGIN IMPORTS #
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
# USER CODE END IMPORTS #

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### UNMODIFIABLE CODE BEGIN ###
DATASET_PATH = Path("/kaggle/working/polymer_Rg/competition")  # path for saving and loading dataset(s)
WORKSPACE_PATH = Path("/kaggle/working/polymer_Rg/output")
PIPELINE_PATH = WORKSPACE_PATH / "pipeline"  # path for saving and loading pipelines
SUBMISSION_PATH = WORKSPACE_PATH / "submission.csv"  # path for saving submission file
EVAL_SET_SIZE = 0.2  # 20% of the data for evaluation
### UNMODIFIABLE CODE END ###

# --- TODO: Update these paths for your specific competition ---
TRAIN_FILE = DATASET_PATH / "train.csv"  # TODO: Replace with your actual filename
TEST_FILE = DATASET_PATH / "test.csv"  # TODO: Replace with your actual filename
SAMPLE_SUBMISSION_FILE = DATASET_PATH / "sample_submission.csv"  # TODO: Replace with your actual filename or None


# USER CODE BEGIN LOAD_DATA #
def load_data():
    train = pd.read_csv(TRAIN_FILE)
    X_test = pd.read_csv(TEST_FILE)
    return train, X_test
# USER CODE END LOAD_DATA #

def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Function to transform data into a format that can be used for training the model.
    Used on both Train and Test data. Test data may initially not contain target columns.
    """

    target_columns = ['Rg']  # TODO: Replace with ACTUAL target columns

    # Separating features and target if present
    data = dataset.copy(deep=True)

    features = data['SMILES'].apply(smiles_to_features)
    valid_smiles = features.notna()
    if not valid_smiles.any():
        raise ValueError("No valid SMILES strings found")
    data = data[valid_smiles].drop(columns=['SMILES'])
    features = features[valid_smiles].apply(pd.Series)
    data = pd.concat([data, features], axis=1)

    has_target = any(col in data.columns for col in target_columns)
    if has_target:
        features = data.drop(columns=target_columns)
        target = data[target_columns].values.flatten()  # Flatten target to 1-D array for regression
    else:
        features = data
        target = None

    return features.values, target


# The main function to orchestrate the data loading, feature engineering, model training and model evaluation
def create_model():
    """
    Function to execute the ML pipeline.
    """
    # USER CODE BEGIN CREATE MODEL #
    # Step 1. Retrieve or load a dataset from hub (if available) or user’s local storage, start path from the DATASET_PATH
    train, X_test = load_data()

    # Step 2. Create a train-test split of the data by splitting the ‘dataset‘ into train_data and test_data.
    # Create a train-validation split
    # Note: EVAL_SET_SIZE is a constant defined above, corresponding to 20% of the data for evaluation
    # Note: You may need to use stratified sampling if the target is categorical
    train_data, eval_test_data = train_test_split(
        train, test_size=EVAL_SET_SIZE, random_state=SEED
    )  # corresponding to 80%, 20% of ‘dataset‘

    train_features, train_target = transform_data(train_data)
    eval_test_features, eval_test_target = transform_data(eval_test_data)
    test_features, _ = transform_data(X_test)

    # Step 3. Train AutoML model. AutoML performs feature engineering and model training.
    model = train_model(train_features, train_target)

    # Step 4. evaluate the trained model using the defined "evaluate_model" function
    model_performance = evaluate_model(model, eval_test_features, eval_test_target)

    # Step 5.  Evaluate predictions for the test dataset
    # **YOU MUST USE automl_predict()**
    predictions: np.ndarray = automl_predict(model, test_features)  # returns 2D array
    output = pd.DataFrame(predictions, columns=['Rg'])  # Adding column name for predictions

    # Adding IDs to output
    output.insert(0, 'id', X_test['id'].values)  # Insert ID column
    output['id'] = output['id'].astype(int)  # Ensure ID column is of integer type

    # USER CODE END CREATE MODEL #
    # If target submission format is not numeric, convert predictions to expected format
    output.to_csv(SUBMISSION_PATH, index=False)
    return model_performance


### UNMODIFIABLE CODE BEGIN ###
def main():
    """
    Main function to execute the ML pipeline.
    """
    print("Files and directories:")
    paths = {
        "Dataset Path": DATASET_PATH,
        "Workspace Path": WORKSPACE_PATH,
        "Pipeline Path": PIPELINE_PATH,
        "Submission Path": SUBMISSION_PATH,
        "Train File": TRAIN_FILE,
        "Test File": TEST_FILE,
        "Sample Submission File": SAMPLE_SUBMISSION_FILE,
    }
    for name, path in paths.items():
        print(f"{name}: {path}")

    model_performance = create_model()
    print("Model Performance on Test Set:", model_performance)


main()

Files and directories:
Dataset Path: /kaggle/working/polymer_Rg/competition
Workspace Path: /kaggle/working/polymer_Rg/output
Pipeline Path: /kaggle/working/polymer_Rg/output/pipeline
Submission Path: /kaggle/working/polymer_Rg/output/submission.csv
Train File: /kaggle/working/polymer_Rg/competition/train.csv
Test File: /kaggle/working/polymer_Rg/competition/test.csv
Sample Submission File: /kaggle/working/polymer_Rg/competition/sample_submission.csv
2025-07-24 00:35:41,608 - ApiComposer - Initial pipeline was fitted in 7.0 sec.
2025-07-24 00:35:41,611 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 35.1 sec.
2025-07-24 00:35:41,613 - AssumptionsHandler - Preset was changed to fast_train due to fit time estimation for initial model.
2025-07-24 00:35:41,624 - ApiComposer - AutoML configured. Parameters tuning: True. Time limit: 1 min. Set of candidate models: ['adareg', 'dtreg', 'knnreg', 'lasso', 'linear', 'normalization', 'pca', 'ransac_lin_

In [6]:
### UNMODIFIABLE IMPORT BEGIN ###
import random
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Tuple
from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import (
    Task,
    TaskTypesEnum,
)  # classification, regression, ts_forecasting.
def train_model(train_features: np.ndarray | pd.DataFrame, train_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(train_features, pd.DataFrame) and isinstance(train_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(train_features, train_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(train_features, np.ndarray) and isinstance(train_target, np.ndarray):
        input_data = InputData.from_numpy(train_features, train_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for train_features and train_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(train_features)} and {type(train_target)}")
        
    model = Fedot(problem=TaskTypesEnum.regression.value,
            timeout=1,
            seed=42,
            cv_folds=5,
            preset='auto',
            metric='mae',
            n_jobs=1,
            with_tuning=True,
            show_progress=True)

    try:
        model.fit(features=input_data) # this is the training step, after this step variable 'model' will be a trained model
    except Exception as e:
        raise RuntimeError(
            f"Model training failed. Please check your data preprocessing carefully. "
            f"Common issues include: missing values, incorrect data types, feature scaling problems, "
            f"or incompatible target variable format. Original error: {str(e)}"
        ) from e

    # Save the pipeline
    pipeline = model.current_pipeline
    pipeline.save(path=PIPELINE_PATH, create_subdir=False, is_datetime_in_path=False)

    return model
def evaluate_model(model: Fedot, test_features: np.ndarray | pd.DataFrame | pd.Series, test_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(test_features, pd.DataFrame) and isinstance(test_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(test_features, np.ndarray) and isinstance(test_target, np.ndarray):
        input_data = InputData.from_numpy(test_features, test_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for test_features and test_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(test_features)} and {type(test_target)}")
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()
def automl_predict(model: Fedot, features: np.ndarray | pd.DataFrame | pd.Series) -> np.ndarray:
    if isinstance(features, (pd.DataFrame, pd.Series)):
        features = features.to_numpy()
    input_data = InputData.from_numpy(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")
    return predictions
def smiles_to_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            "MolWt": Descriptors.MolWt(mol)
            ,"HeavyAtomMolWt": Descriptors.HeavyAtomMolWt(mol)
            ,"NumAtoms": mol.GetNumAtoms()
            ,"MolLogP": Descriptors.MolLogP(mol)
            ,"NumHDonors": Descriptors.NumHDonors(mol)
            ,"NumHAcceptors": Descriptors.NumHAcceptors(mol)
            ,"TPSA": Descriptors.TPSA(mol)
            ,"NumRotatableBonds": Descriptors.NumRotatableBonds(mol)
            ,"RingCount": Descriptors.RingCount(mol)
            ,"NumAromaticRings": Descriptors.NumAromaticRings(mol)
            ,"NumHeteroatoms": Descriptors.NumHeteroatoms(mol)
        }
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fp = mfpgen.GetFingerprint(mol)
        features.update({f'FP_{i}': int(b) for i, b in enumerate(fp)})
        
        return features
    except:
        return None

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

### UNMODIFIABLE IMPORT END ###
# USER CODE BEGIN IMPORTS #
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
# USER CODE END IMPORTS #

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### UNMODIFIABLE CODE BEGIN ###
DATASET_PATH = Path("/kaggle/working/polymer_Tc/competition")  # path for saving and loading dataset(s)
WORKSPACE_PATH = Path("/kaggle/working/polymer_Tc/output")
PIPELINE_PATH = WORKSPACE_PATH / "pipeline"  # path for saving and loading pipelines
SUBMISSION_PATH = WORKSPACE_PATH / "submission.csv"  # path for saving submission file
EVAL_SET_SIZE = 0.2  # 20% of the data for evaluation
### UNMODIFIABLE CODE END ###

# --- TODO: Update these paths for your specific competition ---
TRAIN_FILE = DATASET_PATH / "train.csv"  # TODO: Replace with your actual filename
TEST_FILE = DATASET_PATH / "test.csv"  # TODO: Replace with your actual filename
SAMPLE_SUBMISSION_FILE = DATASET_PATH / "sample_submission.csv"  # TODO: Replace with your actual filename or None


# USER CODE BEGIN LOAD_DATA #
def load_data():
    train = pd.read_csv(TRAIN_FILE)
    X_test = pd.read_csv(TEST_FILE)
    return train, X_test
# USER CODE END LOAD_DATA #

def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Function to transform data into a format that can be used for training the model.
    Used on both Train and Test data. Test data may initially not contain target columns.
    """

    target_columns = ['Tc']  # TODO: Replace with ACTUAL target columns

    # Separating features and target if present
    data = dataset.copy(deep=True)

    features = data['SMILES'].apply(smiles_to_features)
    valid_smiles = features.notna()
    if not valid_smiles.any():
        raise ValueError("No valid SMILES strings found")
    data = data[valid_smiles].drop(columns=['SMILES'])
    features = features[valid_smiles].apply(pd.Series)
    data = pd.concat([data, features], axis=1)

    has_target = any(col in data.columns for col in target_columns)
    if has_target:
        features = data.drop(columns=target_columns)
        target = data[target_columns].values.flatten()  # Flatten target to 1-D array for regression
    else:
        features = data
        target = None

    return features.values, target


# The main function to orchestrate the data loading, feature engineering, model training and model evaluation
def create_model():
    """
    Function to execute the ML pipeline.
    """
    # USER CODE BEGIN CREATE MODEL #
    # Step 1. Retrieve or load a dataset from hub (if available) or user’s local storage, start path from the DATASET_PATH
    train, X_test = load_data()

    # Step 2. Create a train-test split of the data by splitting the ‘dataset‘ into train_data and test_data.
    # Create a train-validation split
    # Note: EVAL_SET_SIZE is a constant defined above, corresponding to 20% of the data for evaluation
    # Note: You may need to use stratified sampling if the target is categorical
    train_data, eval_test_data = train_test_split(
        train, test_size=EVAL_SET_SIZE, random_state=SEED
    )  # corresponding to 80%, 20% of ‘dataset‘

    train_features, train_target = transform_data(train_data)
    eval_test_features, eval_test_target = transform_data(eval_test_data)
    test_features, _ = transform_data(X_test)

    # Step 3. Train AutoML model. AutoML performs feature engineering and model training.
    model = train_model(train_features, train_target)

    # Step 4. evaluate the trained model using the defined "evaluate_model" function
    model_performance = evaluate_model(model, eval_test_features, eval_test_target)

    # Step 5.  Evaluate predictions for the test dataset
    # **YOU MUST USE automl_predict()**
    predictions: np.ndarray = automl_predict(model, test_features)  # returns 2D array
    output = pd.DataFrame(predictions, columns=['Tc'])  # Adding column name for predictions

    # Adding IDs to output
    output.insert(0, 'id', X_test['id'].values)  # Insert ID column
    output['id'] = output['id'].astype(int)  # Ensure ID column is of integer type

    # USER CODE END CREATE MODEL #
    # If target submission format is not numeric, convert predictions to expected format
    output.to_csv(SUBMISSION_PATH, index=False)
    return model_performance


### UNMODIFIABLE CODE BEGIN ###
def main():
    """
    Main function to execute the ML pipeline.
    """
    print("Files and directories:")
    paths = {
        "Dataset Path": DATASET_PATH,
        "Workspace Path": WORKSPACE_PATH,
        "Pipeline Path": PIPELINE_PATH,
        "Submission Path": SUBMISSION_PATH,
        "Train File": TRAIN_FILE,
        "Test File": TEST_FILE,
        "Sample Submission File": SAMPLE_SUBMISSION_FILE,
    }
    for name, path in paths.items():
        print(f"{name}: {path}")

    model_performance = create_model()
    print("Model Performance on Test Set:", model_performance)


main()

Files and directories:
Dataset Path: /kaggle/working/polymer_Tc/competition
Workspace Path: /kaggle/working/polymer_Tc/output
Pipeline Path: /kaggle/working/polymer_Tc/output/pipeline
Submission Path: /kaggle/working/polymer_Tc/output/submission.csv
Train File: /kaggle/working/polymer_Tc/competition/train.csv
Test File: /kaggle/working/polymer_Tc/competition/test.csv
Sample Submission File: /kaggle/working/polymer_Tc/competition/sample_submission.csv
2025-07-24 00:37:49,237 - ApiComposer - Initial pipeline was fitted in 7.5 sec.
2025-07-24 00:37:49,240 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 37.3 sec.
2025-07-24 00:37:49,243 - AssumptionsHandler - Preset was changed to fast_train due to fit time estimation for initial model.
2025-07-24 00:37:49,254 - ApiComposer - AutoML configured. Parameters tuning: True. Time limit: 1 min. Set of candidate models: ['adareg', 'dtreg', 'knnreg', 'lasso', 'linear', 'normalization', 'pca', 'ransac_lin_

In [7]:
### UNMODIFIABLE IMPORT BEGIN ###
import random
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Tuple
from fedot.api.main import Fedot
from fedot.core.data.data import InputData
from fedot.core.repository.tasks import (
    Task,
    TaskTypesEnum,
)  # classification, regression, ts_forecasting.
def train_model(train_features: np.ndarray | pd.DataFrame, train_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(train_features, pd.DataFrame) and isinstance(train_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(train_features, train_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(train_features, np.ndarray) and isinstance(train_target, np.ndarray):
        input_data = InputData.from_numpy(train_features, train_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for train_features and train_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(train_features)} and {type(train_target)}")
        
    model = Fedot(problem=TaskTypesEnum.regression.value,
            timeout=1,
            seed=42,
            cv_folds=5,
            preset='auto',
            metric='mae',
            n_jobs=1,
            with_tuning=True,
            show_progress=True)

    try:
        model.fit(features=input_data) # this is the training step, after this step variable 'model' will be a trained model
    except Exception as e:
        raise RuntimeError(
            f"Model training failed. Please check your data preprocessing carefully. "
            f"Common issues include: missing values, incorrect data types, feature scaling problems, "
            f"or incompatible target variable format. Original error: {str(e)}"
        ) from e

    # Save the pipeline
    pipeline = model.current_pipeline
    pipeline.save(path=PIPELINE_PATH, create_subdir=False, is_datetime_in_path=False)

    return model
def evaluate_model(model: Fedot, test_features: np.ndarray | pd.DataFrame | pd.Series, test_target: np.ndarray | pd.DataFrame | pd.Series):
    if isinstance(test_features, pd.DataFrame) and isinstance(test_target, (pd.DataFrame, pd.Series)):
        input_data = InputData.from_dataframe(test_features, test_target, task=Task(TaskTypesEnum.regression))
    elif isinstance(test_features, np.ndarray) and isinstance(test_target, np.ndarray):
        input_data = InputData.from_numpy(test_features, test_target, task=Task(TaskTypesEnum.regression))
    else:
        raise ValueError("Unsupported data types for test_features and test_target. "
                         "Expected pandas DataFrame and (DataFrame or Series), or numpy ndarray and numpy ndarray."
                         f"Got: {type(test_features)} and {type(test_target)}")
    y_pred = model.predict(features=input_data)
    print("Model metrics: ", model.get_metrics())
    return model.get_metrics()
def automl_predict(model: Fedot, features: np.ndarray | pd.DataFrame | pd.Series) -> np.ndarray:
    if isinstance(features, (pd.DataFrame, pd.Series)):
        features = features.to_numpy()
    input_data = InputData.from_numpy(features, None, task=Task(TaskTypesEnum.regression))
    predictions = model.predict(features=input_data)
    print(f"Predictions shape: {predictions.shape}")
    return predictions
def smiles_to_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        features = {
            "MolWt": Descriptors.MolWt(mol)
            ,"HeavyAtomMolWt": Descriptors.HeavyAtomMolWt(mol)
            ,"NumAtoms": mol.GetNumAtoms()
            ,"MolLogP": Descriptors.MolLogP(mol)
            ,"NumHDonors": Descriptors.NumHDonors(mol)
            ,"NumHAcceptors": Descriptors.NumHAcceptors(mol)
            ,"TPSA": Descriptors.TPSA(mol)
            ,"NumRotatableBonds": Descriptors.NumRotatableBonds(mol)
            ,"RingCount": Descriptors.RingCount(mol)
            ,"NumAromaticRings": Descriptors.NumAromaticRings(mol)
            ,"NumHeteroatoms": Descriptors.NumHeteroatoms(mol)
        }
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)
        fp = mfpgen.GetFingerprint(mol)
        features.update({f'FP_{i}': int(b) for i, b in enumerate(fp)})
        
        return features
    except:
        return None

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator

### UNMODIFIABLE IMPORT END ###
# USER CODE BEGIN IMPORTS #
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
# USER CODE END IMPORTS #

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

### UNMODIFIABLE CODE BEGIN ###
DATASET_PATH = Path("/kaggle/working/polymer_Tg/competition")  # path for saving and loading dataset(s)
WORKSPACE_PATH = Path("/kaggle/working/polymer_Tg/output")
PIPELINE_PATH = WORKSPACE_PATH / "pipeline"  # path for saving and loading pipelines
SUBMISSION_PATH = WORKSPACE_PATH / "submission.csv"  # path for saving submission file
EVAL_SET_SIZE = 0.2  # 20% of the data for evaluation
### UNMODIFIABLE CODE END ###

# --- TODO: Update these paths for your specific competition ---
TRAIN_FILE = DATASET_PATH / "train.csv"  # TODO: Replace with your actual filename
TEST_FILE = DATASET_PATH / "test.csv"  # TODO: Replace with your actual filename
SAMPLE_SUBMISSION_FILE = DATASET_PATH / "sample_submission.csv"  # TODO: Replace with your actual filename or None


# USER CODE BEGIN LOAD_DATA #
def load_data():
    train = pd.read_csv(TRAIN_FILE)
    X_test = pd.read_csv(TEST_FILE)
    return train, X_test
# USER CODE END LOAD_DATA #

def transform_data(dataset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Function to transform data into a format that can be used for training the model.
    Used on both Train and Test data. Test data may initially not contain target columns.
    """

    target_columns = ['Tg']  # TODO: Replace with ACTUAL target columns

    # Separating features and target if present
    data = dataset.copy(deep=True)

    features = data['SMILES'].apply(smiles_to_features)
    valid_smiles = features.notna()
    if not valid_smiles.any():
        raise ValueError("No valid SMILES strings found")
    data = data[valid_smiles].drop(columns=['SMILES'])
    features = features[valid_smiles].apply(pd.Series)
    data = pd.concat([data, features], axis=1)

    has_target = any(col in data.columns for col in target_columns)
    if has_target:
        features = data.drop(columns=target_columns)
        target = data[target_columns].values.flatten()  # Flatten target to 1-D array for regression
    else:
        features = data
        target = None

    return features.values, target


# The main function to orchestrate the data loading, feature engineering, model training and model evaluation
def create_model():
    """
    Function to execute the ML pipeline.
    """
    # USER CODE BEGIN CREATE MODEL #
    # Step 1. Retrieve or load a dataset from hub (if available) or user’s local storage, start path from the DATASET_PATH
    train, X_test = load_data()

    # Step 2. Create a train-test split of the data by splitting the ‘dataset‘ into train_data and test_data.
    # Create a train-validation split
    # Note: EVAL_SET_SIZE is a constant defined above, corresponding to 20% of the data for evaluation
    # Note: You may need to use stratified sampling if the target is categorical
    train_data, eval_test_data = train_test_split(
        train, test_size=EVAL_SET_SIZE, random_state=SEED
    )  # corresponding to 80%, 20% of ‘dataset‘

    train_features, train_target = transform_data(train_data)
    eval_test_features, eval_test_target = transform_data(eval_test_data)
    test_features, _ = transform_data(X_test)

    # Step 3. Train AutoML model. AutoML performs feature engineering and model training.
    model = train_model(train_features, train_target)

    # Step 4. evaluate the trained model using the defined "evaluate_model" function
    model_performance = evaluate_model(model, eval_test_features, eval_test_target)

    # Step 5.  Evaluate predictions for the test dataset
    # **YOU MUST USE automl_predict()**
    predictions: np.ndarray = automl_predict(model, test_features)  # returns 2D array
    output = pd.DataFrame(predictions, columns=['Tg'])  # Adding column name for predictions

    # Adding IDs to output
    output.insert(0, 'id', X_test['id'].values)  # Insert ID column
    output['id'] = output['id'].astype(int)  # Ensure ID column is of integer type

    # USER CODE END CREATE MODEL #
    # If target submission format is not numeric, convert predictions to expected format
    output.to_csv(SUBMISSION_PATH, index=False)
    return model_performance


### UNMODIFIABLE CODE BEGIN ###
def main():
    """
    Main function to execute the ML pipeline.
    """
    print("Files and directories:")
    paths = {
        "Dataset Path": DATASET_PATH,
        "Workspace Path": WORKSPACE_PATH,
        "Pipeline Path": PIPELINE_PATH,
        "Submission Path": SUBMISSION_PATH,
        "Train File": TRAIN_FILE,
        "Test File": TEST_FILE,
        "Sample Submission File": SAMPLE_SUBMISSION_FILE,
    }
    for name, path in paths.items():
        print(f"{name}: {path}")

    model_performance = create_model()
    print("Model Performance on Test Set:", model_performance)


main()

Files and directories:
Dataset Path: /kaggle/working/polymer_Tg/competition
Workspace Path: /kaggle/working/polymer_Tg/output
Pipeline Path: /kaggle/working/polymer_Tg/output/pipeline
Submission Path: /kaggle/working/polymer_Tg/output/submission.csv
Train File: /kaggle/working/polymer_Tg/competition/train.csv
Test File: /kaggle/working/polymer_Tg/competition/test.csv
Sample Submission File: /kaggle/working/polymer_Tg/competition/sample_submission.csv
2025-07-24 00:40:04,441 - ApiComposer - Initial pipeline was fitted in 6.5 sec.
2025-07-24 00:40:04,443 - ApiComposer - Taking into account n_folds=5, estimated fit time for initial assumption is 32.7 sec.
2025-07-24 00:40:04,445 - AssumptionsHandler - Preset was changed to fast_train due to fit time estimation for initial model.
2025-07-24 00:40:04,456 - ApiComposer - AutoML configured. Parameters tuning: True. Time limit: 1 min. Set of candidate models: ['adareg', 'dtreg', 'knnreg', 'lasso', 'linear', 'normalization', 'pca', 'ransac_lin_

In [8]:
import os
import pandas as pd

# Base path where the polymer_* folders are located
base_dir = "/kaggle/working"

# Target list
targets = ["Tg", "FFV", "Tc", "Density", "Rg"]

# Load and merge all result files
merged_df = None
for target in targets:
    result_path = os.path.join(base_dir, f"polymer_{target}", "output", "submission.csv")
    
    df = pd.read_csv(result_path)

    if merged_df is None:
        merged_df = df
    else:
        merged_df = pd.merge(merged_df, df, on="id", how="inner")

# Save final merged result
merged_df.to_csv(os.path.join(base_dir, '/kaggle/working/submission.csv'), index=False)
print("✅ Merged result saved as result.csv")

✅ Merged result saved as result.csv


Done!